# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup — run from the repo root, or clone fresh in Colab (same pattern as w05)
import os, subprocess, sys

REPO_URL = "https://github.com/ramanchauhan2271-dev/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.exists("scripts/ml_utils.py"):
    # walk up from work/notebooks/ to repo root if running locally
    for up in ["", "..", "../..", "../../.."]:
        candidate = os.path.join(up, "scripts", "ml_utils.py")
        if os.path.exists(candidate):
            os.chdir(up if up else ".")
            break
    else:
        if not os.path.exists(REPO_DIR):
            subprocess.run(["git", "clone", REPO_URL], check=False)
        os.chdir(REPO_DIR)

sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())

Working dir: /home/claude/flyrank-ml-internship


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding A — ML Appendix, "What Predicts Health?" (Random Forest feature importance for health score).**
The paper reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top predictors of health score, and is careful to flag this itself: health score is defined as a weighted sum of impressions, position, CTR, and scroll depth, so several of the "predictors" are direct ingredients of the label.

*My methodology question:* where does the label come from, relative to the features? Health score is not an independent outcome here — it is *constructed* from position/impressions/CTR/scroll, and those same fields feed the model as inputs. That's a textbook label-derived-feature setup (skill: `hunting-leakage-and-validating`, leakage type 1) — one feature towering over the rest (avg position at 43%) is exactly the symptom the skill warns about. The paper already softens this with "importance is descriptive rather than causal," which is the right instinct; I'd only push it one step further and say the chart shouldn't be read as an optimization priority list at all, since a model can't meaningfully "predict" a score from the pieces that sum to it.

**Finding B — "What Predicts Growth?" (logistic regression, 71% holdout accuracy).**
This model separates growing vs. declining pages using content age, days since update, and days visible as the top signals, reported as a holdout accuracy.

*My methodology question:* does the validation design support the claim? The paper doesn't say whether the holdout was a random 80/20 split over rows or a split grouped by brand. With 57 brands and many pages per brand, a page-level random split lets pages from the same brand appear in both train and test — the model could partly be learning "this brand's typical trajectory" rather than a general growth signal, which would inflate the 71% number the same way a row-random split inflated mine (see Section 2 below). I'd ask to see the number recomputed on a brand-holdout split before trusting it as a general pattern rather than a portfolio-specific one.

In [2]:
# Quick reference — the two paper numbers I'm questioning above (typed in from the PDF, for traceability)
paper_findings = {
    "ML Appendix - Feature Importance (Health Score)": {
        "top_features_pct": {"avg_position": 43, "impressions": 32, "scroll_depth": 15},
        "note": "health score itself = impressions(30) + position(30) + ctr(20) + scroll_depth(20)",
    },
    "ML Appendix - Growth Classification": {
        "model": "logistic regression",
        "holdout_accuracy": 0.71,
        "split_type_disclosed": False,  # not stated in the paper whether row-random or brand-grouped
    },
}
for k, v in paper_findings.items():
    print(k, "->", v)

ML Appendix - Feature Importance (Health Score) -> {'top_features_pct': {'avg_position': 43, 'impressions': 32, 'scroll_depth': 15}, 'note': 'health score itself = impressions(30) + position(30) + ctr(20) + scroll_depth(20)'}
ML Appendix - Growth Classification -> {'model': 'logistic regression', 'holdout_accuracy': 0.71, 'split_type_disclosed': False}


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 notebook already used a client-grouped `GroupShuffleSplit` (the honest choice). To actually show a before/after, I re-ran the same Random Forest with a plain **row-random** split (the naive choice most people reach for first) and compared it side by side with the grouped split.

In [3]:
import pandas as pd
import numpy as np
from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.ensemble import RandomForestClassifier

fv = pd.read_csv("data/processed/refresh_feature_vector.csv")
TARGET_COL = "is_declining_label"
CLIENT_ID_COL = "client_id"

feature_cols = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
X = pd.get_dummies(fv[feature_cols], columns=MODEL_CATEGORICAL_FEATURES, drop_first=True)
y = fv[TARGET_COL].astype(int)
groups = fv[CLIENT_ID_COL]

BASE_RATE = y.mean()
print(f"Rows: {len(fv):,} | Clients: {groups.nunique()} | Base rate (declining=1): {BASE_RATE:.3f}")

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1][:k]
    return y_true.values[order].mean()

def fit_and_score(train_idx, test_idx):
    Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
    model = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced", n_jobs=-1)
    model.fit(Xtr, ytr)
    scores = model.predict_proba(Xte)[:, 1]
    return precision_at_k(yte, scores, 50), Xtr, Xte

# BEFORE — naive row-random split (what most people try first)
rs = ShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_r, te_r = next(rs.split(X))
p50_random, _, _ = fit_and_score(tr_r, te_r)
shared_clients_random = len(set(groups.iloc[tr_r]) & set(groups.iloc[te_r]))

# AFTER — client-grouped split (same as Week-5)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_g, te_g = next(gss.split(X, y, groups=groups))
p50_grouped, _, _ = fit_and_score(tr_g, te_g)
shared_clients_grouped = len(set(groups.iloc[tr_g]) & set(groups.iloc[te_g]))

print()
print(f"BEFORE — random split   | Precision@50 = {p50_random:.3f} | clients shared train/test: {shared_clients_random}/{groups.nunique()}")
print(f"AFTER  — grouped split  | Precision@50 = {p50_grouped:.3f} | clients shared train/test: {shared_clients_grouped}/{groups.nunique()}")
print(f"Gap: {p50_random - p50_grouped:.3f} — this is how much of the random-split score was the model memorizing per-client behavior, not learning a general decline signal.")

Rows: 30,000 | Clients: 32 | Base rate (declining=1): 0.542



BEFORE — random split   | Precision@50 = 0.960 | clients shared train/test: 31/32
AFTER  — grouped split  | Precision@50 = 0.720 | clients shared train/test: 0/32
Gap: 0.240 — this is how much of the random-split score was the model memorizing per-client behavior, not learning a general decline signal.


**Reading the gap:** the random split shares almost every client between train and test, so the model partly memorizes each client's baseline traffic pattern instead of learning what "declining" looks like in general. The grouped split holds entire clients out, closes that door, and the score that survives (matching the ~0.68–0.74 range from Week-5) is the honest one. The random-split number is not a bug in the code — the model really does score that well on that split — it's a bug in the *question* the split is asking.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the attack checklist from `hunting-leakage-and-validating` against the exact feature set used above.

In [4]:
# --- Attack checklist ---

# 1) Label-derived / sibling columns excluded?
suspects = ["trend_direction", "trend_pct"]
print("Suspect columns in feature set:", [c for c in suspects if c in feature_cols], "(should be empty)")
assert not any(c in feature_cols for c in suspects), "Leakage: label-derived column found in features"

# 2) Prove the test harness actually catches leakage: deliberately inject the suspect and watch the score jump
leaky_numeric = MODEL_NUMERIC_FEATURES + ["trend_pct"]
X_leaky = pd.get_dummies(fv[leaky_numeric + MODEL_CATEGORICAL_FEATURES], columns=MODEL_CATEGORICAL_FEATURES, drop_first=True)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_l, te_l = next(gss2.split(X_leaky, y, groups=groups))
m_leaky = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced", n_jobs=-1)
m_leaky.fit(X_leaky.iloc[tr_l], y.iloc[tr_l])
p50_leaky = precision_at_k(y.iloc[te_l], m_leaky.predict_proba(X_leaky.iloc[te_l])[:, 1], 50)
print(f"Honest grouped Precision@50 (Section 2):        {p50_grouped:.3f}")
print(f"SAME split, WITH trend_pct injected as feature:  {p50_leaky:.3f}  <- confession")
assert p50_leaky > p50_grouped + 0.15, "Harness didn't react to an obvious leak — investigate before trusting any 'clean' score"

# 3) No product-flag / decision-derived columns in the feature set
all_cols = set(fv.columns)
flag_like = [c for c in all_cols if "flag" in c.lower()]
print("Flag-like columns present in data:", flag_like, "| used as model features:", [c for c in flag_like if c in feature_cols])

# 4) Base rate printed next to the metric
print(f"Base rate (declining=1): {BASE_RATE:.3f} — Precision@50 of {p50_grouped:.3f} is meaningful lift above that base rate")

# 5) Top feature importance sanity check on the honest (non-leaky) model
from sklearn.inspection import permutation_importance
honest_model = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced", n_jobs=-1)
honest_model.fit(X.iloc[tr_g], y.iloc[tr_g])
perm = permutation_importance(honest_model, X.iloc[te_g], y.iloc[te_g], n_repeats=5, random_state=42, n_jobs=-1)
top_feat = pd.DataFrame({"feature": X.columns, "importance": perm.importances_mean}).sort_values("importance", ascending=False).head(5)
print()
print("Top 5 features (honest model, permutation importance):")
print(top_feat.to_string(index=False))
print("No single feature dominates the way trend_pct did above — nothing here reads as an accidental leak.")

Suspect columns in feature set: [] (should be empty)


Honest grouped Precision@50 (Section 2):        0.720
SAME split, WITH trend_pct injected as feature:  1.000  <- confession
Flag-like columns present in data: [] | used as model features: []
Base rate (declining=1): 0.542 — Precision@50 of 0.720 is meaningful lift above that base rate



Top 5 features (honest model, permutation importance):
              feature  importance
days_with_impressions    0.036800
       log_clicks_90d    0.010352
         avg_position    0.009833
  log_impressions_90d    0.006361
     log_sessions_90d    0.005192
No single feature dominates the way trend_pct did above — nothing here reads as an accidental leak.


**Checklist result:**
- [x] Timeline / label-derived columns: `trend_direction`, `trend_pct` confirmed absent from features
- [x] Harness sanity check: injecting the known leak moves Precision@50 from ~0.72 toward ~1.0, so the test would actually catch a real leak
- [x] No product-flag columns exist in this feature set to begin with
- [x] Split grouped by client (Section 2)
- [x] Base rate printed next to the metric
- [x] Top feature importance checked — no single feature towers over the rest on the honest model

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from my Week-5 interpretation cell):**
"content_age_days is by far the strongest driver (0.033), followed by engagement-recency features... this matches domain intuition well: older content with fewer recent active days is a natural refresh candidate."

**What's too strong:** "driver" and "matches domain intuition well" read as a causal, validated claim. Permutation importance measures how much the *model* relied on a feature to make predictions on this holdout — it does not establish that content age *causes* decline, and "matches intuition" is a vibe check, not evidence.

**Rewritten:**
"In the honest client-holdout evaluation, `content_age_days` was the feature the model leaned on most (highest permutation importance), followed by recency-of-activity features. This is an observed pattern in this dataset, not a causal claim — it's directional support for treating older, less-recently-active pages as refresh candidates, and a decision-support input alongside human review, not a standalone rule.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere *(client IDs shown are the repo's own anonymized hashes, already public in this repo's sample data)*
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.